# 한국어 가스라이팅 이진 분류기 학습 (v1)

**Task**: 유저 메시지 → 조작(1) / 정상(0)

**데이터**:
- Positive (조작): eoh9 가스라이팅 대화 CSV의 `competition` 컬럼 (1,699 unique)
- Negative (정상): songys ChatbotData CSV (11,823)

**모델**: `klue/roberta-base` (한국어 SOTA, 110M)

**사용법**:
1. Colab 열기 → 파일 → 노트북 업로드 → 이 ipynb 선택
2. 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
3. 좌측 파일 패널에 3개 업로드 (drag-and-drop):
   - `gaslighting_dialogues.csv`
   - `chatbot_data.csv`
   - `sequences.jsonl` (우리 21 케이스 평가용)
4. 런타임 → 모두 실행 (또는 위에서부터 Shift+Enter)
5. 결과 확인 + 마지막 셀로 가중치 다운로드

**예상 시간**: T4 GPU에서 5~10분 (5 epochs)

## 1. 환경 확인 + 의존성

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory:', torch.cuda.get_device_properties(0).total_memory / 1e9, 'GB')
else:
    print('!!! GPU 사용 X. 런타임 유형을 T4 GPU로 변경하세요.')

In [ ]:
!pip install -q transformers datasets scikit-learn pandas

## 2. 데이터 로드 + 라벨링

In [ ]:
import pandas as pd
import os

# 업로드된 파일 확인
required = ['gaslighting_dialogues.csv', 'chatbot_data.csv', 'sequences.jsonl']
for f in required:
    if not os.path.exists(f):
        raise FileNotFoundError(f'{f} 누락. 좌측 파일 패널에 업로드 필요.')
    print(f'OK: {f} ({os.path.getsize(f) / 1024:.1f} KB)')

In [ ]:
# Positive (조작): 가스라이팅 CSV의 competition 컬럼
df_gas = pd.read_csv('gaslighting_dialogues.csv')
print('가스 raw:', len(df_gas), 'rows')
print('컬럼:', df_gas.columns.tolist())

# competition 컬럼만 + 중복 제거
gas_texts = df_gas['competition'].dropna().drop_duplicates().tolist()
print(f'가스 unique: {len(gas_texts)}개')
print('샘플:', gas_texts[:3])

In [ ]:
# Negative (정상): ChatbotData. label 1/2 → 0 변환 (전부 정상으로 사용)
df_cb = pd.read_csv('chatbot_data.csv')
print('Chatbot raw:', len(df_cb), 'rows')
print('컬럼:', df_cb.columns.tolist())

# Q 컬럼만 사용 (유저 메시지 톤에 가까움)
normal_texts = df_cb['Q'].dropna().drop_duplicates().tolist()
print(f'정상 unique: {len(normal_texts)}개')
print('샘플:', normal_texts[:3])

In [ ]:
# 통합 + 라벨링
data = pd.DataFrame({
    'text': gas_texts + normal_texts,
    'label': [1] * len(gas_texts) + [0] * len(normal_texts)
})
data = data.sample(frac=1, random_state=42).reset_index(drop=True)  # 셔플
print(f'전체: {len(data)}개')
print(f'  조작(1): {(data.label == 1).sum()}')
print(f'  정상(0): {(data.label == 0).sum()}')
print(f'  imbalance ratio: {(data.label == 0).sum() / (data.label == 1).sum():.2f}:1')

In [ ]:
# train/test split (stratified)
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    data, test_size=0.2, stratify=data.label, random_state=42
)
print(f'train: {len(train_df)} (조작 {(train_df.label==1).sum()} / 정상 {(train_df.label==0).sum()})')
print(f'test:  {len(test_df)} (조작 {(test_df.label==1).sum()} / 정상 {(test_df.label==0).sum()})')

## 3. 토크나이저 + 모델 (KLUE/RoBERTa-base)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = 'klue/roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = model.cuda() if torch.cuda.is_available() else model
print('Model loaded:', MODEL_NAME, '| params:', sum(p.numel() for p in model.parameters()) / 1e6, 'M')

In [ ]:
# Dataset 클래스
from torch.utils.data import Dataset, DataLoader

MAX_LEN = 128
BATCH_SIZE = 32

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_ds = TextDataset(train_df.text.tolist(), train_df.label.tolist(), tokenizer)
test_ds = TextDataset(test_df.text.tolist(), test_df.label.tolist(), tokenizer)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
print(f'train batches: {len(train_loader)}, test batches: {len(test_loader)}')

## 4. 학습

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm

EPOCHS = 5
LR = 2e-5
device = 'cuda' if torch.cuda.is_available() else 'cpu'

optimizer = AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f'epoch {epoch+1}/{EPOCHS}')
    for batch in pbar:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(**batch)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += out.loss.item()
        pbar.set_postfix(loss=f'{out.loss.item():.4f}')
    print(f'epoch {epoch+1} avg loss: {total_loss / len(train_loader):.4f}')

## 5. Test set 평가

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='test'):
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        preds = out.logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(batch['labels'].cpu().numpy().tolist())

print('Test Accuracy:', accuracy_score(all_labels, all_preds))
print()
print('Classification report:')
print(classification_report(all_labels, all_preds, target_names=['정상(0)', '조작(1)'], digits=4))
print('Confusion matrix:')
print(confusion_matrix(all_labels, all_preds))

## 6. 우리 21 케이스 평가 (진짜 일반화 측정)

In [ ]:
import json

with open('sequences.jsonl', 'r', encoding='utf-8') as f:
    cases = [json.loads(line) for line in f if line.strip()]

# 각 user turn 추출 + expected label (정상=0, 조작=1)
eval_turns = []
for case in cases:
    is_manipulation = not case['case_id'].startswith('normal_')
    for turn in case['turns']:
        if turn['role'] == 'user':
            eval_turns.append({
                'case_id': case['case_id'],
                'category': case['category'],
                'text': turn['content'],
                'expected_label': 1 if is_manipulation else 0,
            })

print(f'평가 turn: {len(eval_turns)}개')
print(f'  조작: {sum(1 for t in eval_turns if t["expected_label"]==1)}')
print(f'  정상: {sum(1 for t in eval_turns if t["expected_label"]==0)}')

In [ ]:
# Inference
model.eval()
results = []
with torch.no_grad():
    for turn in eval_turns:
        enc = tokenizer(turn['text'], max_length=MAX_LEN, padding='max_length', truncation=True, return_tensors='pt')
        enc = {k: v.to(device) for k, v in enc.items()}
        out = model(**enc)
        probs = torch.softmax(out.logits, dim=-1).cpu().numpy()[0]
        pred = int(probs.argmax())
        results.append({
            **turn,
            'predicted_label': pred,
            'prob_manipulation': float(probs[1]),
            'correct': pred == turn['expected_label'],
        })

# 출력
import pandas as pd
res_df = pd.DataFrame(results)
print(f'21 케이스 정확도: {res_df.correct.mean():.3f} ({res_df.correct.sum()}/{len(res_df)})')
print()
print('case_id별 결과:')
for _, row in res_df.iterrows():
    mark = 'OK ' if row.correct else 'FAIL'
    print(f'  [{mark}] {row.case_id:40s} | expected={row.expected_label} pred={row.predicted_label} prob_m={row.prob_manipulation:.3f}')
    print(f'         user: {row.text[:80]}')

In [ ]:
# 카테고리별 분석
print('카테고리별 정확도:')
for cat in res_df.category.unique():
    sub = res_df[res_df.category == cat]
    print(f'  {cat:30s}: {sub.correct.mean():.2f} ({sub.correct.sum()}/{len(sub)})')
print()
print('정상 vs 조작 정확도:')
for lbl, name in [(0, '정상'), (1, '조작')]:
    sub = res_df[res_df.expected_label == lbl]
    print(f'  {name}: {sub.correct.mean():.3f} ({sub.correct.sum()}/{len(sub)})')

In [ ]:
# 결과 저장
res_df.to_csv('eval_21cases_results.csv', index=False, encoding='utf-8')
with open('eval_21cases_results.jsonl', 'w', encoding='utf-8') as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')
print('저장: eval_21cases_results.csv / .jsonl')

## 7. 모델 가중치 저장 + 다운로드

In [ ]:
OUTPUT_DIR = './gaslight_klue_roberta_v1'
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'저장 완료: {OUTPUT_DIR}')

# zip으로 압축
import shutil
shutil.make_archive('gaslight_klue_roberta_v1', 'zip', OUTPUT_DIR)
print('압축 완료: gaslight_klue_roberta_v1.zip')

In [ ]:
# 다운로드 (Colab)
from google.colab import files
files.download('gaslight_klue_roberta_v1.zip')
files.download('eval_21cases_results.csv')
files.download('eval_21cases_results.jsonl')